# Module 2 — Chunking Strategies and HyDE
**Created by Shivank Agarwal**

Before a RAG system can search a long document, it cuts the document into smaller pieces called **chunks**.
How you cut decides what the search can find later.

| Part | What you will try |
|---|---|
| 2.1 | Does chunk **size** change what we retrieve? (small vs medium vs large) |
| 2.2 | **Fixed-length** chunking |
| 2.3 | **Semantic** chunking (cut where the meaning changes) |
| 2.4 | **Structure-based** chunking (cut at Markdown headings) |
| 2.5 | **HyDE** — search with a *fake answer* instead of the question |

You need an OpenAI API key (put it in the `.env` file, or paste it when asked below).

In [ ]:
# Run this once if the libraries are not installed yet (remove the # first):
# %pip install -q -r ../../00_Setup/requirements.txt

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from a .env file if you have one
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---
## 2.1 Finding a good chunk size
We cut the same short article into chunks of **50**, **150** and **400** characters,
store each version in its own vector database, and ask the same question to all three.

* Too small → each chunk is a fragment, the answer is spread over many pieces
* Too large → one chunk holds many topics, the match is blurry
* Just right → one chunk holds one complete idea

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

document_text = """
The human heart is a muscular organ that pumps blood through the circulatory system.
This blood delivers oxygen and nutrients to the body and removes metabolic waste. The heart
has four chambers: the right atrium, the right ventricle, the left atrium, and the left ventricle.
The average adult heart beats about 60 to 100 times per minute. The circulatory system consists of the heart,
blood vessels, and blood. It is a vital part of the human body, responsible for transporting essential substances.
A healthy lifestyle, including regular exercise and a balanced diet, is crucial for cardiovascular health.
"""

# Wrap the text in a LangChain Document (text + metadata)
documents = [Document(page_content=document_text, metadata={"source": "heart_article"})]

chunk_sizes = [50, 150, 400]   # small, medium, large (in characters)
chunked_documents = {}         # size -> list of chunks

for size in chunk_sizes:
    # ----- TODO 1: Split the document with RecursiveCharacterTextSplitter (10% overlap) and store the chunks -----
    # Hint: RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.1))
    # Hint: splitter.split_documents(documents) returns a list of smaller Documents
    chunks = []  # replace with your code
    chunked_documents[size] = chunks
    print(f"chunk_size={size:<4} -> {len(chunks)} chunks")

print("\nThe medium (150) chunks look like this:")
for i, chunk in enumerate(chunked_documents.get(150, []), start=1):
    print(f"  [{i}] {chunk.page_content!r}")

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# OpenAI embedding model: turns each chunk into a list of 1,536 numbers
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_dbs = {}  # size -> Chroma vector store

for size, chunks in chunked_documents.items():
    # ----- TODO 2: Store the chunks of this size in their own Chroma collection -----
    # Hint: Chroma.from_documents(chunks, embeddings, collection_name=..., ids=...)
    # Hint: Use a different collection_name per size, e.g. f"heart_chunks_{size}"
    # Hint: ids like f"{size}-{i}" stop duplicates if you run the cell twice
    db = None  # replace with your code
    vector_dbs[size] = db
    print(f"Stored {len(chunks)} chunks for size {size}")

In [ ]:
query = "What is the primary function of the heart and its connection to the circulatory system?"
k = 3  # how many chunks to bring back

print(f"Query: {query}\n")
for size, db in vector_dbs.items():
    print("-" * 70)
    print(f"Chunk size {size}")
    # ----- TODO 3: Retrieve the k most similar chunks from this vector store -----
    # Hint: db.similarity_search(query, k=...) returns a list of Documents
    # Hint: A small store may have fewer than k chunks, so use min(k, len(chunked_documents[size]))
    retrieved_docs = []  # replace with your code
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"  Result {i}: {doc.page_content.strip()}")

**Look at the output:** with size 50 the answer is chopped into fragments; with size 400 the whole
article comes back as one block. Size 150 usually gives the cleanest, most focused answer for this text.

---
## 2.2 Fixed-length chunking
The simplest strategy: cut every N characters.
We use a small Markdown article about Machine Learning for parts 2.2 to 2.4.

In [ ]:
import textwrap
from langchain_text_splitters import CharacterTextSplitter

# textwrap.dedent removes the common left indentation so headings start with "#"
text = textwrap.dedent("""
    # Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed.

    ## Types of Machine Learning

    ### Supervised Learning
    Supervised learning uses labeled training data to learn a mapping function from input to output. Common examples include:
    - Classification: Predicting categories (spam/not spam)
    - Regression: Predicting continuous values (house prices)

    ### Unsupervised Learning
    Unsupervised learning finds hidden patterns in data without labeled examples:
    - Clustering: Grouping similar data points
    - Dimensionality reduction: Reducing feature space

    ### Reinforcement Learning
    An agent learns through interaction with an environment, receiving rewards or penalties for actions taken.
""").strip()

# (a) Pure fixed length: separator="" means "cut anywhere", even in the middle of a word
naive_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0, separator="")
naive_chunks = naive_splitter.split_text(text)
print("Pure fixed-length (first 3 chunks) - notice the broken words:")
for chunk in naive_chunks[:3]:
    print("   ", repr(chunk))

In [ ]:
# ----- TODO 4: Fixed-length chunks of up to 200 characters that only cut at line breaks, with 20 characters overlap -----
# Hint: CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
# Hint: .split_text(text) returns a list of strings
chunks = []  # replace with your code

print("Fixed-length chunks that respect line breaks:")
for i, chunk in enumerate(chunks, start=1):
    print(f"Chunk {i} (length: {len(chunk)}):")
    print(chunk)
    print("-" * 40)

---
## 2.3 Semantic chunking
Instead of counting characters, the **SemanticChunker** embeds every sentence and cuts where the meaning
of one sentence is far from the next one (a "topic jump").

> You may see a yellow `DeprecationWarning` saying `langchain-experimental` is being retired.
> That is only a notice from the library. The code still works, so carry on.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

# ----- TODO 5: Create a SemanticChunker with OpenAI embeddings and split the text into Documents -----
# Hint: SemanticChunker(OpenAIEmbeddings(model="text-embedding-3-small"))
# Hint: .create_documents([text]) returns a list of Documents
docs = []  # replace with your code

for i, doc in enumerate(docs, start=1):
    print(f"Chunk {i}:")
    print(doc.page_content)
    print("-" * 40)

---
## 2.4 Structure-based chunking (Markdown headers)
Many real documents already have structure: headings, sections, sub-sections.
`MarkdownHeaderTextSplitter` cuts at the headings **and** remembers which heading each chunk came from
(in the metadata). That metadata is very useful later for filtering and for showing sources.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# ----- TODO 6: Split the Markdown text at the headings listed above -----
# Hint: MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
# Hint: .split_text(text) returns Documents that carry the headings in .metadata
md_header_splits = []  # replace with your code

for i, doc in enumerate(md_header_splits, start=1):
    print(f"Chunk {i}:")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)

---
## 2.5 HyDE — Hypothetical Document Embeddings
**Problem:** a shopper types *"I want running shoes"*. Our product titles say *trainers*, *sneakers*,
*marathon*, *Ultraboost*... the words do not match the question.

**HyDE idea:** first ask an LLM to *imagine* what a good answer would look like (a fake, "hypothetical"
document), then search with that fake answer. The fake answer uses the same kind of words as the real
documents, so the search finds them.

To make the effect easy to see, we search with **BM25** (classic keyword search, like a search engine
from before AI). The same trick works with embeddings.

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_openai import ChatOpenAI

shopper_query = "I want running shoes"
products = [
    "Adidas Ultraboost sneakers for men , Marathon Shoes",
    "Nike Air Zoom Pegasus trainers",
    "Puma casual footwear for everyday wear",
    "Woodland leather trekking boots",
    "Asics Gel-Kayano stability trainers",
    "Reebok CrossFit gym footwear",
    "Clarks formal leather boots",
    "Skechers lightweight walking sneakers",
    "New Balance Fresh Foam trail trainers",
    "Bata budget-friendly casual footwear",
    "Fila sports footwear for gym workouts",
    "Converse high-top casual sneakers",
]

# BM25 works on words, so we lower-case and split every product title into words
tokenized_products = [p.lower().split() for p in products]
bm25 = BM25Okapi(tokenized_products)

# --- Search WITHOUT HyDE: use the shopper's words directly -------------------
scores_no_hyde = bm25.get_scores(shopper_query.lower().split())
top_no_hyde = [i for i in np.argsort(scores_no_hyde)[::-1] if scores_no_hyde[i] > 0.1][:3]

print("Top products WITHOUT HyDE:")
if not top_no_hyde:
    print("  - No relevant results found")
for i in top_no_hyde:
    print(f"  - {products[i]}  (score: {scores_no_hyde[i]:.2f})")

In [ ]:
# Chat model used to write the hypothetical answer.
# reasoning_effort="none" -> fast, direct answers (no long "thinking")
llm = ChatOpenAI(model="gpt-6-luna", reasoning_effort="none")

# ----- TODO 7: Ask the LLM to write a hypothetical answer (fake product listings) for the shopper's query -----
# Hint: Write a prompt like: "Write 4 short product titles from an online shoe store that match: <query>"
# Hint: llm.invoke(prompt).text gives you the text answer
hyde_answer = ""  # replace with your code

print("Hypothetical answer written by the LLM:\n")
print(hyde_answer)

In [ ]:
# ----- TODO 8: Search again with the query + hypothetical answer and print the top 3 products -----
# Hint: hyde_query = shopper_query + " " + hyde_answer
# Hint: Lower-case and split it into words, then bm25.get_scores(words)
# Hint: np.argsort(scores)[::-1][:3] gives the indexes of the 3 best products
scores_hyde = np.zeros(len(products))  # replace with your code

top_hyde = np.argsort(scores_hyde)[::-1][:3]
print("Top products WITH HyDE:")
for i in top_hyde:
    print(f"  - {products[i]}  (score: {scores_hyde[i]:.2f})")

**What happened?** The shopper said *running shoes*; the catalogue says *trainers*, *Ultraboost*,
*Pegasus*, *Gel-Kayano*. The LLM's fake answer is full of those catalogue-style words,
so the search now finds the real running products.